# pytRIBS

## The Need For pytRIBS
pytRIBS was designed to aid users of the TIN-based Real-time Integrated Basin Simulator ([tRIBS](https://tribshms.readthedocs.io/en/latest/)) distributed hydrologic model in model setup, execution, and result analysis. Prior to pytRIBS, setting up a tRIBS model could take multiple days, involved a number of proprietary software programs, and was prone to user error. pytRIBS addresses all of these challenges, letting users approach hydrologic modeling in a programmatic and efficient manner.

![tRIBS Workflow](../smf_assets/tRIBS_workflow_horz.png)

**Figure 1.** A tRIBS workflow consists of three main steps: (1) Collate and generate model inputs. (2) Run the model simulation(s). (3) Review and analyze results. The first and last step (denoted by red text and dashed lines) are user intensive activities that challenge reproducibility and are often very time consuming. The asterisks denote steps required for the parallel operation of the model. 

## pytRIBS Design

!['pytRIBS Design'](../smf_assets/pytRIBS_design.png)

**Figure 2.** pytRIBS uses an object-oriented approach to capture the major components and steps required for setting up, simulating, and analyzing a tRIBS model. The preprocessing classes are intended to expedite the first step in setting up a tRIBS model, whereas the simulation classes provide users with tools to effectively run and analyze the model through a Python interface. The Project class is not directly linked to another class as it is limited to storing directory information and meta data. Only select attributes or methods are shown for each class, for a full list of attributes and methods see the associated documentation. 
*Water balance can be calculated for both the basin averaged condition or at individual nodes. 
**Mesh class relies on Preprocess and MeshGenerator classes accessed via these instances.

# Sandbox Start
pytRIBS has a set of workflows and general functions that can be used help creating a tRIBS model. The workflows are generally focused around downloading and processing commonly used datasets like the NLDAS-2 workflow for meteorological data. The general functions while helpful in any scenario are very helpful if datasets not built into pytRIBS are being used. A general model development workflow with pytRIBS is as follows:
1. Setup projection information and generate directory structure.
2. Watershed delineation.
3. Computational mesh development.
4. Soil data processing.
5. Land use data processing.
7. Meteorological data processing.
8. tRIBS input file generation.

The steps above are what is contained within this notebook. With the exception of providing a pre-generated model mesh. There are different ways to run a model after it has been setup with pytRIBS. For this sandbox environment the tRIBS code has already been downloaded and compiled. Thus, the model can be ran from within the Run_Model notebook in this same directory.

To get started with the sandbox we have to first import the Pytohn packages we will need like any other Python code.


## Imports

In [274]:
# (TL) Imports main pytRIBS classes needed to build the project,
# mesh, soils, land cover, meteorological setup, and final model input files

# note you can install pytRIBS via pip; see: https://pypi.org/project/pytRIBS/
from pytRIBS.classes import *

In [275]:
# (TL) Imports general PYthon libraries the notebook uses. 
# Gives the notebook its support tools for moving files around, 
# reading geospatial data, and making diagnostic plots.

# if you have installed pytRIBS, the following libraries should already be in your environment
import os, sys, shutil
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from shapely.ops import unary_union
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from pathlib import Path
import json
import pandas as pd
from rasterio.features import geometry_mask
from shapely.geometry import mapping

In [276]:
# (TL) Plot switches for calibration workflow.
# Set these to False when doing repeated calibration runs.

make_mesh_plots = False
make_soil_plots = False              # Also calculates percentage of soil classes

In [ ]:
# (TL) Calibration run controls.
# Change this one block for each new calibration run.

# -------------------------
# RUN ID NAMING
# -------------------------
location = "SMF"
event_date = "20140812"

# My convention:
# 00–09 = baseline/setup
# 10–19 = soil hydraulic parameters, single-variable runs
# 20–29 = channel loss runs
# 30–39 = routing/timing runs
# 40+   = multi-variable calibration runs
run_number = "45"
change_tested = "cc6000_cv2"

run_id = f"{location}_{event_date}_{run_number}_{change_tested}"

assert len(run_number) == 2 and run_number.isdigit(), "run_number should be two digits, like '00', '10', '20', '30', or '40'"
# -------------------------
# EVENT / SIMULATION SETTINGS
# -------------------------
start_date = "08/01/2014/00/00"
runtime_hours = 450
rain_interval_hours = 0.25

# Window used later in Run_Model for focused hydrograph comparison.
event_start = "2014-08-12 16:00"
event_end = "2014-08-13 12:00"

# -------------------------
# SOIL CALIBRATION CONTROLS
# -------------------------
Ks_mult = 1.0
f_mult = 1.0

# Advanced soil anisotropy controls
# These correspond to the As and Au fields in the tRIBS soil table.
# Keep both at 1.0 unless intentionally testing anisotropy.
As_value = 1.0
Au_value = 1.0

# -------------------------
# CHANNEL LOSS CONTROLS
# -------------------------
optpercolation = 0                  # 0 indicates channel transmission losses are turned off; set to 1 to test
channelconductivity_mmhr = 6000       # Baseline is 70
channelporosity = 0.4               # Baseline is 0.4

# -------------------------
# ROUTING CONTROLS
# -------------------------
kinemvelcoef = 2                    # Baseline is 3
flowexp = 0.3                       # Baseline is 0.3
channelroughness = 0.04             # Baseline is 0.04   This is the Manning's roughness coeffieient. 
                                    # A lower number means a smoother channel. Typical values fall between 0.01 and 0.07, but can go higher for dense vegetation. 
channelwidthcoeff = 2.33            # Baseline is 2.33

# -------------------------
# ORGANIZED OUTPUT FOLDERS
# -------------------------
notebook_dir = Path.cwd()

# If this notebook is running inside smf_demo, project_root is one folder up.
# If not, project_root is the current folder.
project_root = notebook_dir.parent if notebook_dir.name == "smf_demo" else notebook_dir

calib_dir = project_root / "calibration_work"

def get_run_category(run_number):
    """Assign calibration run category based on the full run number."""
    run_num = int(run_number)

    if 0 <= run_num <= 9:
        return "00_baseline"
    elif 10 <= run_num <= 19:
        return "10_soilKs"
    elif 20 <= run_num <= 29:
        return "20_channel_loss"
    elif 30 <= run_num <= 39:
        return "30_routing"
    elif run_num >= 40:
        return "40_multivariable"
    else:
        return "99_other"

run_category = get_run_category(run_number)

run_input_dir = calib_dir / "01_run_inputs" / run_category
run_results_dir = calib_dir / "02_results" / run_category / run_id
csv_export_dir = calib_dir / "03_comparisons" / "csv_exports"
plot_export_dir = calib_dir / "03_comparisons" / "hydrograph_plots"
summary_export_dir = calib_dir / "03_comparisons" / "summary_tables"
log_dir = calib_dir / "06_logs"

for folder in [
    run_input_dir,
    run_results_dir,
    csv_export_dir,
    plot_export_dir,
    summary_export_dir,
    log_dir
]:
    folder.mkdir(parents=True, exist_ok=True)

# File paths for this run
input_file_abs = run_input_dir / f"{run_id}.in"
log_file_abs = log_dir / f"{run_id}.log"
output_prefix_abs = run_results_dir / run_id

# Relative paths are safer for tRIBS because the notebook/model is run from smf_demo.
input_file = os.path.relpath(input_file_abs, notebook_dir)
log_file = os.path.relpath(log_file_abs, notebook_dir)
output_prefix = os.path.relpath(output_prefix_abs, notebook_dir)

# Save current run information so Run_Model.ipynb can read it.
current_run_config = {
    "location": location,
    "event_date": event_date,
    "run_number": run_number,
    "change_tested": change_tested,
    "run_id": run_id,
    "run_category": run_category,
    "start_date": start_date,
    "runtime_hours": runtime_hours,
    "rain_interval_hours": rain_interval_hours,
    "event_start": event_start,
    "event_end": event_end,
    "Ks_mult": Ks_mult,
    "f_mult": f_mult,
    "As_value": As_value,
    "Au_value": Au_value,
    "optpercolation": optpercolation,
    "channelconductivity_mmhr": channelconductivity_mmhr,
    "channelporosity": channelporosity,
    "kinemvelcoef": kinemvelcoef,
    "flowexp": flowexp,
    "channelroughness": channelroughness,
    "channelwidthcoeff": channelwidthcoeff,
    "input_file": input_file,
    "log_file": log_file,
    "output_prefix": output_prefix,
    "csv_export_dir": os.path.relpath(csv_export_dir, notebook_dir),
    "plot_export_dir": os.path.relpath(plot_export_dir, notebook_dir),
    "summary_export_dir": os.path.relpath(summary_export_dir, notebook_dir)
}

config_path = calib_dir / "current_run_config.json"
config_path.write_text(json.dumps(current_run_config, indent=2))

print(f"Current run ID: {run_id}")
print(f"Run category: {run_category}")
print(f"Input file will be written to: {input_file}")
print(f"Results prefix will be: {output_prefix}")
print(f"Run config saved to: {config_path}")

Current run ID: SMF_20140812_45_cc6000_cv2
Run category: 40_multivariable
Input file will be written to: ../calibration_work/01_run_inputs/40_multivariable/SMF_20140812_45_cc6000_cv2.in
Results prefix will be: ../calibration_work/02_results/40_multivariable/SMF_20140812_45_cc6000_cv2/SMF_20140812_45_cc6000_cv2
Run config saved to: /workspaces/tRIBS-Pima-Canyon/workspaces/SMF_Calibration_pytRIBS/calibration_work/current_run_config.json


## Project Class: Setup Model Information

In [278]:
# (TL) Sets up notebook’s identity. Keep name as SMF because some forcing files use this base name.

name = location   # location is currently "SMF" from the calibration-control cell
epsg = 26912
proj = Project(os.getcwd(), name, epsg)

print(f"Base model name: {name}")
print(f"Calibration run ID: {run_id}")

Base model name: SMF
Calibration run ID: SMF_20140812_45_cc6000_cv2


In [279]:
# # (TL) Sets up notebook’s identity. Sets case name and projection, then creates 
# # the main Project object that stores directory paths and metadata for the model setup.

# name ='SMF'
# epsg = 26912
# proj = Project(os.getcwd(),name,epsg) # Create an instance of a Project Class

# # (TL) os.getcwd() means “get current working directory” so pytRIBS 
# # will build the project relative to whatever folder the notebook is running from

The output below is the folder directory structure that pytRIBS generates where the tRIBS model data is stored.

In [280]:
# (TL) Displays the project directory structure so you can see
# where pytRIBS will story different model components and outputs.

proj.directories

{'model': 'data/model',
 'preprocessing': 'data/preprocessing',
 'results': 'results',
 'soil': 'data/model/soil',
 'land': 'data/model/land',
 'met_precip': 'data/model/met/precip',
 'met_meteor': 'data/model/met/meteor',
 'mesh': 'data/model/mesh'}

In [281]:
# (TL) Displays the project metadata to confirm the model name, 
# coordinate system, and related setup information.

proj.meta

{'Name': 'SMF', 'Scenario': None, 'EPSG': 26912}

The code below code copies in the needed pre-processed data sets and sets file paths to required data.

In [282]:
# (TL) Sets the DEM path and copies the preprocessed land-use and soil rasters into the project 
# directory structure so later model components can reference local project copies.

# Set the file path to our DEM
dem = '../smf_init_data/USGS_10m_clip.tif'

# Copy over our pre-generated Soil and Land Use ID rasters into our project directories
landuse_ras = '../smf_init_data/LandUse.asc'
shutil.copy(landuse_ras, proj.directories['land'])
landuse_ras = f"{proj.directories['land']}/{os.path.basename(landuse_ras)}"

soil_ras = '../smf_init_data/ADOT_SoilTypes.asc'
shutil.copy(soil_ras, proj.directories['soil'])
soil_ras = f"{proj.directories['soil']}/{os.path.basename(soil_ras)}"

## Mesh Class: Process DEM and Generate Mesh

### Preprocessing
Here initiate the class with arguments for the Preprocessing and MeshGeneration component classes. 

Before we make a mesh we need to delineate the watershed and to do that we use the code below to define some of information required.

In [283]:
# (TL) Defines watershed preprocessing settings: outlet location, snapping distance, 
# and stream-initation threshold area, then packages them for the mesh workflow.

# Preprocessing Arguments
verbose_mode=False# suppresses whitebox output

# UTM coordinates for outlet/pour point
x = 394456
y = 3686772

snap_dis = 100 # allowable distance in m for snapping outlet points to stream network

# This area threshold for determining streams for watershed delineation
# Lets initially try 0.1sqkm, this value is important because it determines the number of streams in tRIBS mesh
threshold_area = 2e4 # area in m^2 for determining stream network

# tuple of preprocessing arguments to be passed into the the mesh class
preprocess_args = ([x,y],snap_dis, threshold_area, dem, verbose_mode,proj.meta,proj.directories['preprocessing'])

In [284]:
# Defines output file paths for the clipped DEM watershed boundary, 
# stream network, and outlet, and sets the maximum mesh refinement level.

# Mesh Generation Arguments
# output directory can be specified in Preproceesing, but if not provided preprocessing is the default directory 
output_dir = proj.directories['preprocessing'] 

path_to_raster = f'{output_dir}/{name}_clipped_ext.tif' # these are default outputs but can be further modified. See documentation.
path_to_watershed = f'{output_dir}/{name}_boundary.shp'
path_to_stream_network = f'{output_dir}/{name}_stream.shp'
path_to_outlet = f'{output_dir}/{name}_outlet.shp'
maxlevel= 8 # This can be set to none, if so the maximum level possible will be used. 
    #(TL) Higher values allow finer detail

mesh_generation_args = (path_to_raster, path_to_watershed, path_to_stream_network, path_to_outlet, maxlevel)

The code below executes the watershed delineation and saves all of the results to the filepaths listed above.

In [285]:
# (TL) Creates the mesh object and executes the preprocessing/mesh setup workflow 
# using the DEM, outlet, stream threshold, and mesh settings

tmesh = Mesh(preprocess_args=preprocess_args, generate_mesh_args=mesh_generation_args,meta=proj.meta)

Could not create output settings.json file. WBT is likely installed somewhere without write permission.


Now, we have created all the files necessary for generating a mesh. Below we provide a visualization of these data. Note you could read this data in independently then plot it, but they are all ready assigned to ```tmesh.mesh_generator``` or alternatively can be used from the MeshGeneration class.

#### Example Figure 1: Preprocessing products produced by pytRIBS

In [286]:
# (TL) Plots the DEM/preprocessing output for visual quality control.
# Skipped during calibration runs unless make_mesh_plots = True.

if make_mesh_plots:

    dem_data = tmesh.mesh_generator.data
    valid_data = dem_data[dem_data > -500] 
    vmin, vmax = np.percentile(valid_data, [2, 98])
    print(f"Plotting with elevation limits: {vmin:.1f}m to {vmax:.1f}m")

    fig, ax = plt.subplots(figsize=(10, 8))

    extent = tmesh.mesh_generator.get_extent()

    img = ax.imshow(
        dem_data,
        extent=extent,
        cmap='terrain',
        vmin=vmin,
        vmax=vmax 
    )

    cbar = fig.colorbar(img, ax=ax, orientation='vertical', shrink=0.8)
    cbar.set_label('Elevation (m)', fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    tmesh.mesh_generator.watershed.plot(
        ax=ax,
        facecolor='none',
        edgecolor='red',
        linewidth=2,
        zorder=2
    )

    tmesh.mesh_generator.stream_network.plot(
        ax=ax,
        linewidth=1.5,
        color='blue',
        zorder=2
    )

    tmesh.mesh_generator.outlet.plot(
        ax=ax,
        color='yellow',
        marker='*',
        markersize=200,
        edgecolor='black',
        zorder=3
    )

    ax.set_xlabel('Easting (UTM)', fontsize=14)
    ax.set_ylabel('Northing (UTM)', fontsize=14)
    ax.set_title('Watershed, Stream Network, and Outlet', fontsize=16)
    ax.tick_params(axis='both', which='major', labelsize=12)

    legend_elements = [
        Line2D([0], [0], color='red', lw=2, label='Watershed Boundary'),
        Line2D([0], [0], color='blue', lw=1.5, label='Stream Network'),
        Line2D([0], [0], color='yellow', marker='*', markeredgecolor='black', 
               markersize=15, linestyle='None', label='Outlet')
    ]

    ax.legend(handles=legend_elements, loc='upper left', fontsize=12, framealpha=0.9)

    minx, miny, maxx, maxy = tmesh.mesh_generator.watershed.total_bounds
    width = maxx - minx
    height = maxy - miny

    buffer_pct = 0.01
    ax.set_xlim(minx - (width * buffer_pct), maxx + (width * buffer_pct))
    ax.set_ylim(miny - (height * buffer_pct), maxy + (height * buffer_pct))

    plt.tight_layout()
    plt.show()

else:
    print("Skipping DEM/preprocessing QC plot.")

Skipping DEM/preprocessing QC plot.


### Mesh Generation
With the data from previous steps we can generate a locally refined mesh rapidly and easily using a Harr wavelet transform. Below is an example for creating the create the points file required for a tRIBS model simulation if that option is selected in tRIBS input file.

In [287]:
# (TL) Sets the mesh-detail threshold that controls how many significant terrain points 
# are kept; lower values create a denser, more detailed mesh. 

# This parameter is the most important piece of the mesh generation in pytRIBS.
# It controls how many points will be classified as significant thus kept versus in-significant points that don't need to be included in the mesh.
# Lower values means more points kept (denser mesh). General range from 0 to 1.
threshold = 0.5

In [288]:
# (TL) Extracts significant terrain points for mesh generation and creates 
# a buffered watershed polygon for later workflows.

# note we'll use buffered watershed for other workflows to ensure data exists outside of the mesh
points, buffered_watershed = tmesh.mesh_generator.extract_points_from_significant_details(threshold)

In [289]:
# Adds an extra 500 m buffer around the watershed to later 
# datasets/workflows extend beyond the exact basin boundary.

buffered_watershed = buffered_watershed.buffer(250*2)

In [290]:
# (TL) Saves the buffershed watershed as a shapefile so it can be reused for later 
# GIS or model-preprocessign steps.

# save out the buffered watershed in case you need it later
gdf = gpd.GeoDataFrame({'id':[1]}, geometry=[buffered_watershed], crs=tmesh.meta['EPSG'])
gdf.to_file(f"{proj.directories['preprocessing']}/{name}_buffered_watershed.shp")

In [291]:
# (TL) Converts the selected mesh points into a GeoDataFrame and writes 
# the .points file that tRIBS uses as mesh input. 


# Write out points the step above to a point file that tRIBS can read.
tmesh.pointfilename['value'] = proj.directories['mesh']+'/'+name+'.points'
gdf = tmesh.mesh_generator.convert_points_to_gdf(points)
tmesh.mesh_generator.write_point_file(gdf,tmesh.pointfilename['value'])

## Soil Class: Obtain and Generate tRIBS Soil Parameters

There are two methods to provide tRIBS soil parameters: a soil ID map and table or gridded parameter rasters. For SMF since we have the ADOT soil data as polygons we will use the simpler method, a soil ID map with a lookup table. Note pytRIBS has functanility to download gridded soil datasets that are publicly available but not used in this exercise.

In [292]:
# (TL) Creates the Soil object that will store 
# all soil-related inputs and parameters for the model.

soil = Soil(meta=proj.meta)

Another component of the soil class is the initial ground water table and depth to bedrock.

Bedrock data can be quite difficult to find but it is becoming more available from Digital Soil Mapping (DSM) products. For this model we will apply the [SOLUS](https://www.nrcs.usda.gov/resources/data-and-reports/soil-landscapes-of-the-united-states-solus) dataset which has already been pre-processed for use here. Due to the shallow soils here and the dry conditions we are assuming that the initial groundwater table is essentially at the bedrock. To do that we multiplied our depth to bedrock raster by 95% i.e. if the deptht to berock is 1m then the initial groundwater depth is 0.05m above the bedrock.

All we need to do is copy over the data into our model directory.

In [293]:
# (TL) Copies the bedrock-depth and initial-groundwater rasters into the project soil folder and links 
# them to the Soil object. *Set assumptions of initial gw table.*
# These rasters define important subsurface starting conditions.

# Depth to Bedrock
shutil.copy('../smf_init_data/SOLUS_Bedrock_m.asc', proj.directories['soil'])
soil.bedrockfile['value'] = f"{proj.directories['soil']}/SOLUS_Bedrock_m.asc"

# Initial Groundwater Table
shutil.copy('../smf_init_data/InitGW_95pct_mm.asc', proj.directories['soil'])
soil.gwaterfile['value'] = f"{proj.directories['soil']}/InitGW_95pct_mm.asc"

Now that the soil class is created we can tell pytRIBS that we already have the soil data.

In [294]:
# (TL) LInks the Soil object to the soil-class raster and to the soil parameter table (.sdt), which 
# together tell the model where each soil type occurs and what properties it has. This cell does 
# not yet assign the actual hydraulic values. It just sets up the file connections.  

# Give file path of soil ID map we already copied into our directory structure at the start
soil.soilmapname['value'] = soil_ras

# We already have a blank soil data table so we just need to copy it over and link it to the soil class in pytRIBS
soil_table = '../smf_init_data/soils.sdt'
shutil.copy(soil_table, proj.directories['soil'])
soil.soiltablename['value'] = f"{proj.directories['soil']}/soils.sdt"

In [295]:
# (TL) Reads the soil table and soil ID raster into memory so the notebook can 
# inspect soil classes, assign parameter values, and make plots. 

soil_table = soil.read_soil_table(textures=True)
soil_map = InOut.read_ascii(soil.soilmapname['value'])

After we have read in the soil table, we can see the associated class ID and texture as follows.

In [296]:
# (TL) Loops through the soil table and prints the soil class IDs and textures so you 
# can confirm which soil categories are present in the class case. 

# Lets printout the class IDs to see what they are. They are set to the MUSYM IDs from the ADOT dataset.
for cls in soil_table:
    print(f"Class ID and texture: {cls['ID']}, {cls['Texture']}")

Class ID and texture: 1, RS
Class ID and texture: 2, CO
Class ID and texture: 3, CeD
Class ID and texture: 4, EbD
Class ID and texture: 5, Cb


In [297]:
# (TL) SUPER IMPORTANT: Defines hydraulic and thermal soil parameters by soil class, 
# then assigns those values into the soil table. 
# This is one of the main cells controlling infiltration and runoff behavior. 

# Lets define the soil parameters. Some are parameters are specific to each soil class, some will be static across all classes
# More information here: https://tribshms.readthedocs.io/en/latest/man/Model_Parameters_Forcings.html

# Define Hydraulic Parameters for Specific Soil Classes
# Here we map the specific soil IDs found in the ADOT dataset (MUSYM) to physical hydraulic parameters.
# In a research setting, these might be derived from pedotransfer functions (like Rosetta) or field tests.
# For this exercise, we are using the values from the ADOT data as a starting point.

# Parameters:
# - Ks: Saturated hydraulic conductivity (mm/hr)
    # This is an important calibration parameter
# - thetaS: Saturation soil moisture content (porosity) (-)
# - thetaR: Residual soil moisture content (-)
# - m: Pore size distribution index (Lambda) (-)
# - PsiB: Air entry/Bubbling pressure (mm)
# - f: Conductivity decay parameter (1/mm) - Controls how fast Ksat decreases with depth
    # This is an important calibration parameter
# - n: Porosity (-)

# Define a dictionary to map your ADOT Soil IDs to parameters.
soil_param_lookup = {
    # Soil: RS
    '1': {
        'Ks': 3.6,      # From ADOT dataset, KEY calibration parameter
        'thetaS': 0.40,  # From ADOT dataset
        'thetaR': 0.06,  # Initial guess we are using 0.5*wilting point
        'm': 0.38,       # Value selected from Rawls et al 1984.
        'PsiB': -390,    # From ADOT dataset
        'f': 0.02,       # Assume initial value, KEY calibraiton parameter
        'n': 0.40        # Initial guess we are using this is equal to thetaS
    },
    # Soil: CO
    '2': {'Ks': 2.8,  'thetaS': 0.40, 'thetaR': 0.05, 'm': 0.25, 'PsiB': -401, 'f': 0.002, 'n': 0.40},
    # Soil: CeD
    '3': {'Ks': 6.6,  'thetaS': 0.40, 'thetaR': 0.05, 'm': 0.25, 'PsiB': -183, 'f': 0.002, 'n': 0.40},
    # Soil: EbD
    '4': {'Ks': 1.0,  'thetaS': 0.42, 'thetaR': 0.10, 'm': 0.20, 'PsiB': -450, 'f': 0.002, 'n': 0.42},
    # Soil: Cb
    '5': {'Ks': 17.3, 'thetaS': 0.39, 'thetaR': 0.03, 'm': 0.18, 'PsiB': -117, 'f': 0.001, 'n': 0.39},
}

# Iterate through the soil table and assign values based on the ID lookup
# We also set the constant thermal/anisotropy parameters here.

for cls in soil_table:
    # Constant Parameters (Same for all soils in this exercise)
    # As and Au are anisotropy ratio fields used by the tRIBS soil table.
    # We expose them in the top calibration-control cell so they can be tested deliberately.
    cls['As'] = As_value
    cls['Au'] = Au_value
    cls['ks'] = 0.7     # volumetric heat conductivity, J/msK
    cls['Cs'] = 1.4e6   # soil heat capacity, J/m^3k
    
    # --- Spatially Varying Hydraulic Parameters ---
    # Get the ID from the current row (ensure it's a string to match our dict keys)
    current_id = str(cls['ID'])
    
    if current_id in soil_param_lookup:
        params = soil_param_lookup[current_id]
        
        # Assign values to the pytRIBS object
        cls['Ks'] = params['Ks'] * Ks_mult
        cls['thetaS'] = params['thetaS']
        cls['thetaR'] = params['thetaR']
        cls['m'] = params['m']
        cls['PsiB'] = params['PsiB']
        cls['f'] = params['f'] * f_mult
        cls['n'] = params['n']
        
        print(f"Assigned parameters for Soil ID {current_id} ({cls['Texture']})")
        
    else:
        # Fallback in case something goes wrong
        print(f"WARNING: Soil ID {current_id} not found in lookup dictionary! Using defaults.")
        cls['Ks'] = 10.0
        cls['thetaS'] = 0.4
        cls['thetaR'] = 0.05
        cls['m'] = 0.2
        cls['PsiB'] = -200
        cls['f'] = 0.001
        cls['n'] = 0.001

# (TL) Write the soil table for this calibration run.
# This writes the updated soil parameters to the normal pytRIBS soil table first,
# then saves a run-specific copy so each calibration run is reproducible.

# Always use the standard working soil table as the temporary write location.
# This prevents SameFileError if the cell is rerun.
working_soil_table = Path("data/model/soil/soil.sdt")

# Write the populated table to the normal pytRIBS working location
soil.write_soil_table(soil_table, str(working_soil_table), textures=True)

# Save a run-specific copy in calibration_work
run_specific_soil_abs = run_input_dir / f"soils_{run_id}.sdt"

if working_soil_table.resolve() != run_specific_soil_abs.resolve():
    shutil.copy(working_soil_table, run_specific_soil_abs)
else:
    print("Working soil table and run-specific soil table are the same file; skipping copy.")

# Point the final tRIBS input file to the run-specific soil table
soil.soiltablename['value'] = os.path.relpath(run_specific_soil_abs, notebook_dir)

print(f"Working soil table written to: {working_soil_table}")
print(f"Run-specific soil table saved to: {run_specific_soil_abs}")
print(f"Final model will use soil table: {soil.soiltablename['value']}")
print(f"Ks multiplier: {Ks_mult}")
print(f"f multiplier: {f_mult}")
print(f"As anisotropy ratio: {As_value}")
print(f"Au anisotropy ratio: {Au_value}")

# What this cell does: print the final calibrated soil parameters by soil ID.

soil_audit_rows = []

for cls in soil_table:
    current_id = str(cls["ID"])
    
    if current_id in soil_param_lookup:
        params = soil_param_lookup[current_id]
        
        final_Ks = cls["Ks"]
        final_f = cls["f"]
        
        f_depth_scale_mm = 1 / final_f if final_f != 0 else np.nan
        
        soil_audit_rows.append({
            "Soil ID": current_id,
            "Texture": cls.get("Texture", ""),
            "Baseline Ks (mm/hr)": params["Ks"],
            "Ks multiplier": Ks_mult,
            "Final Ks (mm/hr)": final_Ks,
            "Baseline f (1/mm)": params["f"],
            "f multiplier": f_mult,
            "Final f (1/mm)": final_f,
            "Approx 1/f depth scale (mm)": f_depth_scale_mm,
            "Approx 1/f depth scale (m)": f_depth_scale_mm / 1000 if pd.notna(f_depth_scale_mm) else np.nan,
            "As": cls["As"],
            "Au": cls["Au"],
        })

soil_audit_table = pd.DataFrame(soil_audit_rows)

display(soil_audit_table)

Assigned parameters for Soil ID 1 (RS)
Assigned parameters for Soil ID 2 (CO)
Assigned parameters for Soil ID 3 (CeD)
Assigned parameters for Soil ID 4 (EbD)
Assigned parameters for Soil ID 5 (Cb)
Working soil table written to: data/model/soil/soil.sdt
Run-specific soil table saved to: /workspaces/tRIBS-Pima-Canyon/workspaces/SMF_Calibration_pytRIBS/calibration_work/01_run_inputs/40_multivariable/soils_SMF_20140812_45_cc6000_cv2.sdt
Final model will use soil table: ../calibration_work/01_run_inputs/40_multivariable/soils_SMF_20140812_45_cc6000_cv2.sdt
Ks multiplier: 1.0
f multiplier: 1.0
As anisotropy ratio: 1.0
Au anisotropy ratio: 1.0


,Soil ID,Texture,Baseline Ks (mm/hr),Ks multiplier,Final Ks (mm/hr),Baseline f (1/mm),f multiplier,Final f (1/mm),Approx 1/f depth scale (mm),Approx 1/f depth scale (m),As,Au
0,1,RS,3.6,1.0,3.6,0.020,1.0,0.020,50.0,0.05,1.0,1.0
1,2,CO,2.8,1.0,2.8,0.002,1.0,0.002,500.0,0.50,1.0,1.0
2,3,CeD,6.6,1.0,6.6,0.002,1.0,0.002,500.0,0.50,1.0,1.0
3,4,EbD,1.0,1.0,1.0,0.002,1.0,0.002,500.0,0.50,1.0,1.0
4,5,Cb,17.3,1.0,17.3,0.001,1.0,0.001,1000.0,1.00,1.0,1.0


Now that we have the soil table lets tell tRIBS that in input file we will be using option 0 that corresponds to using the soil table not gridded parameter rasters

In [298]:
# (TL) Sets the soil-input option for tRIBS/pytRIBSs so the model uses a soil-class map + soil table approach, 
# rather than some more fully distributed parameter option.  

soil.optsoiltype['value'] = 0

The last soil step is for setting up the channel transmission losses. There are a couple ways to set this up in tRIBS. For this exercise we will use the Constant Loss method.

This method requires 3 input parameters:

    - CHANNELCONDUCTIVITY      # steady-state bed conductivity [m/s]
    - CHANNELPOROSITY          # channel bed porosity [-]
    - CHANNELWIDTH             # channel width [m]

These parameter values are the same values for all of the reaches across the entire model. So assumptions will need to be made like with the channel width.

(TL) The notebook introduces channel transmission losses at the end of the soil discussion, but the actual parameter assignment occurs later in the Model section. In the baseline class case, transmission losses are effectively turned off (optpercolation = 0), even though conductivity and porosity values are defined. So this section is more of a prepared option than an active part of the initial run.

#### Example Figure 3: Example soil classification map generated from pytRIBS Soil Class

In [299]:
# (TL) Plots soil raster/classes for QC.
# (Do soil classes look plausible? 
# Does the watershed outline line up correctly? 
# Are any areas missing or masked unexpectedly? 
# Are the class IDs the ones your soil table expects?)
# Skipped during calibration runs unless make_soil_plots = True.

if make_soil_plots: 

    data = soil_map['data']
    transform = soil_map['profile']['transform']

    # Calculate Extent (Standard GeoTransform logic)
    x_min = transform[2]
    x_max = x_min + soil_map['profile']['width'] * transform[0]
    y_max = transform[5]
    y_min = y_max + soil_map['profile']['height'] * transform[4] 
    extent = [x_min, x_max, y_min, y_max]

    # Mask  Data
    masked_data = np.ma.masked_less(data, 0)

    # Find unique soil classes for the legend
    unique_classes = np.unique(masked_data.compressed())
    num_classes = len(unique_classes)

    print(f"Found {num_classes} unique soil classes: {unique_classes}")

    # Create Discrete Colormap
    cmap = plt.cm.get_cmap('cividis', num_classes)
    norm = mcolors.BoundaryNorm(np.arange(len(unique_classes) + 1), cmap.N) 

    # Plotting
    fig, ax = plt.subplots(figsize=(12, 10))
    img = ax.imshow(
        masked_data,
        extent=extent,
        cmap=cmap,
        interpolation='nearest' # Prevents blurring between classes
    )

    # 6. Watershed Boundary
    tmesh.mesh_generator.watershed.plot(
        ax=ax, 
        facecolor='none', 
        edgecolor='red', 
        linewidth=2,
        label='Watershed Boundary'
    )

    # 7. Custom Colorbar for Categorical Data
    # We tell the colorbar to only show ticks at the center of each discrete color
    cbar = plt.colorbar(img, ax=ax, ticks=unique_classes)
    cbar.set_label('Soil Class ID', fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    # Set the limits of the image to the watershed bounds
    bounds = tmesh.mesh_generator.watershed.total_bounds
    ax.set_xlim([bounds[0], bounds[2]])
    ax.set_ylim([bounds[1], bounds[3]])

    # Labels
    ax.set_xlabel('Easting (UTM)', fontsize=14)
    ax.set_ylabel('Northing (UTM)', fontsize=14)
    ax.set_title(f'Soil Classification Map', fontsize=16)

    plt.tight_layout()
    plt.show()

else:
    print("Skipping soil raster QC plot.")


Skipping soil raster QC plot.


In [300]:
# (TL) What this cell does: calculate soil ID percentages only if the soil map was created.
# If the soil map was not created in this run, skip without error.

soil_percent_table = None

required_objects = ["soil_map", "soil_table", "tmesh"]
missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    print("Skipping soil percent table.")
    print("These required object(s) were not created in this run:")
    print(", ".join(missing_objects))

else:
    try:
        from rasterio.features import geometry_mask
        from shapely.geometry import mapping

        # Get soil raster data and profile
        data = soil_map["data"]
        profile = soil_map["profile"]
        transform = profile["transform"]

        # If raster data has a band dimension, use the first band
        if data.ndim == 3:
            data = data[0]

        # Get watershed geometry
        watershed_gdf = tmesh.mesh_generator.watershed

        # Create a mask where True = inside watershed
        inside_watershed = geometry_mask(
            [mapping(geom) for geom in watershed_gdf.geometry],
            out_shape=data.shape,
            transform=transform,
            invert=True,
            all_touched=False
        )

        # Exclude nodata values if nodata is defined
        nodata = profile.get("nodata", None)

        valid_soils = inside_watershed & np.isfinite(data)

        if nodata is not None:
            valid_soils = valid_soils & (data != nodata)

        # Match the earlier plotting logic: ignore negative IDs
        valid_soils = valid_soils & (data >= 0)

        soil_values = data[valid_soils]

        if len(soil_values) == 0:
            print("No valid soil pixels found inside the watershed.")
            soil_percent_table = None

        else:
            # Count pixels by soil ID
            soil_counts = pd.Series(soil_values).value_counts().sort_index()

            soil_percent_table = pd.DataFrame({
                "Soil ID": soil_counts.index.astype(int),
                "Pixel Count": soil_counts.values,
                "Percent of Watershed": 100 * soil_counts.values / soil_counts.values.sum()
            })

            # Add texture names if soil_table has them
            try:
                texture_lookup = {
                    int(cls["ID"]): cls.get("Texture", "")
                    for cls in soil_table
                }
                soil_percent_table["Texture"] = soil_percent_table["Soil ID"].map(texture_lookup)
                soil_percent_table = soil_percent_table[
                    ["Soil ID", "Texture", "Pixel Count", "Percent of Watershed"]
                ]
            except Exception:
                print("Soil ID percentages calculated, but texture names could not be added.")

            display(soil_percent_table)

    except Exception as e:
        print("Skipping soil percent table because something needed for the calculation was unavailable.")
        print(f"Reason: {e}")

,Soil ID,Texture,Pixel Count,Percent of Watershed
0,1,RS,33817,64.756233
1,2,CO,12193,23.348397
2,3,CeD,5112,9.788978
3,4,EbD,1020,1.953200
4,5,Cb,80,0.153192


## Met Class:
Meteorological forcing for the tRIBS model can be supplied either as point-based station data or as raster data. Managing these data manually can be time-consuming and susceptible to errors. To simplify this process, the Met class allows users to specify the filepaths to already generated forcings files or efficiently obtain meteorological forcing data from the North American Land Data Assimilation System for a specified location and time period. 

For this notebook forcing data has already been prepared so we will not be using the NLDAS workflow. First we need to tell pytRIBS the files paths where our pre-processed forcing will be located:

In [301]:
# (TL) Creates the meteorology component of the model; 
# actual forcing files are assumed to already exist from Generate_Met_Forcing

met = Met(meta=proj.meta) # Initializing the pytRIBS Met Class
met.hydrometbasename['value'] = name
met.hydrometstations['value'] = "../smf_init_data/met/Master_Met.sdf" # File path to where station data file will live
met.gaugestations['value'] = "../smf_init_data/met/Master_Precip.sdf" # File path to where station data file will live

Meteorological forcings for SMF have already been generated using the `Generate_Met_Forcing` notebook.

If you are interested in changing the simulation length or time period of the simulation you can rerun 
that notebook but it is recommended to run this notebook first entirely through to generate the main input file.
With the input file you can directly edit the main input file with the file paths to the new forcing data.

More details on the required tRIBS forcing files are located here: [Model Forcings](https://tribshms.readthedocs.io/en/latest/man/Model_Parameters_Forcings.html#model-forcings)

## Land Class: Create Land Cover Map and Assign tRIBS Land Cover Parameters
tRIBS supports spatially varying land cover parameters, which can be provided through a classification map and table, raster data, or a combination of both. These inputs may remain static over time or vary across different periods (e.g., monthly, seasonally, annually). The Land class offers various attributes and methods to manage these parameters, although the availability and resolution of land cover data often differ significantly between applications, necessitating more user input. Consequently, the Land class does not offer an all-encompassing workflow but instead provides a set of tools to manage data from various sources and helper functions to generate the necessary input files for a tRIBS simulation. 

For this notebook we have already pre-procssed the National Land Cover Database dataset (NLCD) from 2014 for the watershed. Which we already copied into the land folder at the start of this notebook. The map is broken down into `3` classes:  
`1 - South Facing Slopes`  
`2 - North Facing Slopes`  
`3 - Roadway`

While the NLCD only has the `2` land cover classes (shrub/scrub and the roadway) the shrub/scrub class was further separated into 2 classes becuase of what was observed during field visits to the site.

We will setting initial vegetation parameter based on based tRIBs project. More details on the required tRIBS parameters is located here: [Model Parameters](https://tribshms.readthedocs.io/en/latest/man/Model_Parameters_Forcings.html#model-parameters)

In [302]:
# (TL) Create the model component that will store and manage land-cover info 
# and land surface parameters

# Initialize the pytRIBS Land class
land = Land(meta=proj.meta)

In [303]:
# (TL) Links the land object to the land-cover raster and land parameter table. The raster defines the 
# spatial distribution of land classes, and the .ldt file defines the parameter values associated with each class.

# Tell pytRIBS where the landcover map that we already copied over exists
land.landmapname['value'] = f"{proj.directories['land']}/LandUse.asc"
land.landtablename['value'] = f"{proj.directories['land']}/land_use_params.ldt"

In [304]:
# (TL) Defines the land-use parameter dictionary that pytRIBS will use to build the .ldt land 
# parameter table. This is where each land-cover class is assigned its model parameter values, 
# so it is the main place where land-cover assumptions enter the simulation.

# Similar to how we set up the Soil Table we will make a dictionary of 
# parameter values then use the pytRIBS writing function

# Define the Land Use Parameter Dictionary
land_param_lookup = {
    # Class 1: South Facing Slopes (e.g., Scrub/Shrub)
    '1': {
        'P': 0.4,       # Sparse canopy, lots of throughfall
        'S': 1.5,       # Low storage
        'K': 0.12,      # Drainage Coeff
        'b2': 4.7,      # Drainage Exp
        'Al': 0.18,     # Lower albedo (darker)
        'h': 1,       # Short vegetation
        'Kt': 0.4,      # Optical transmission
        'Rs': 120,      # High resistance (desert plants)
        'V': 0.15,       # Vegetation Fraction 
        'LAI': 1.5,     # Low Leaf Area Index
        'theta*_s': 0.37, # Soil Mositure Stress threshold
        'theta*_t': 0.30
    },
    # Class 2: North Facing Slopes 
    '2': {
        'P': 0.4,       # Sparse canopy, lots of throughfall
        'S': 1.5,       # Low storage
        'K': 0.12,      # Drainage Coeff
        'b2': 4.7,      # Drainage Exp
        'Al': 0.18,     # Lower albedo (darker)
        'h': 1,       # Short vegetation
        'Kt': 0.4,      # Optical transmission
        'Rs': 120,      # High resistance (desert plants)
        'V': 0.30,       # Vegetation Fraction (60% bare soil)
        'LAI': 1.5,     # Low Leaf Area Index
        'theta*_s': 0.37, # Stress threshold
        'theta*_t': 0.30
    },
    # Class 3: Roadway / Developed (Impervious)
    '3': {
        'P': 0.99,       # No canopy, rain hits ground instantly
        'S': 0.01,       # No storage
        'K': 0.001, 
        'b2': 0.001, 
        'Al': 0.15,     # Asphalt/Concrete albedo
        'h': 0.01,      # Near zero height
        'Kt': 0.99,     # All light hits ground
        'Rs': 9999,     # Infinite resistance (no transpiration)
        'V': 0.01,      # No vegetation
        'LAI': 0.01, 
        'theta*_s': 0.37, 
        'theta*_t': 0.30
    },
}

# Construct the list of dictionaries for the write function
landuse_list = []

for lu_id, params in land_param_lookup.items():
    # Create a copy so we don't mess up the original dict
    row = params.copy()
    
    # Add the ID to the row (required by the write function)
    row['ID'] = lu_id
    
    # Add dummy interception Parameters (a and b1) for interception scheme we are not using.
    row['a'] = -9999
    row['b1'] = -9999
    
    landuse_list.append(row)

# Write the Table
land.write_landuse_table(landuse_list, land.landtablename['value'])

print(f"Land Use Table written to: {land.landtablename['value']}")

Land Use Table written to: data/model/land/land_use_params.ldt


## Model Class: Pre-flight Check  and Model Simulation
The Model Class allows users to modify tRIBS model inputs, validate that all inputs are appropriate for the chosen options, and run the tRIBS model directly using Docker and the Docker SDK for Python.  It can be initialized with or without preprocessing classes, providing a flexible approach to numerical experiments, and also supports manual setup or starting through an existing input file.

For this notebook we will only be using the Model Class to setup the input file. As the sandbox environment is a Linux environment the tRIBs model can be ran directly instead of using Docker. In this same directory there is another notebook called `run_tRIBS.ipynb` that is used to run the model setup developed here.

In [305]:
# (TL) Combines the met, land, soil, mesh, and metadata objects into one full 
# model object. This is the main handoff from separate setup sections to final model configuration. 


model = Model(met=met,land=land,soil=soil,mesh=tmesh,meta=proj.meta)

The last set of parameters we need to define is for setting up the channel transmission losses. There are a couple ways to set this up in tRIBS. For this exercise we will use the Constant Loss method.

This method requires 3 input parameters:

    - CHANNELCONDUCTIVITY is the steady-state bed conductivity [m/s]
    - CHANNELPOROSITY is the channel bed porosity [-]

These parameter values are the same values for all of the reaches across the entire model.

Or you can test the model with no transmission losses at all by turning off the option.

In [306]:
# (TL) Channel transmission loss settings for this calibration run.

model.optpercolation['value'] = optpercolation
model.channelconductivity['value'] = channelconductivity_mmhr / 3.6e6
model.channelporosity['value'] = channelporosity

print(f"optpercolation: {optpercolation}")
print(f"channelconductivity: {channelconductivity_mmhr} mm/hr = {model.channelconductivity['value']} m/s")
print(f"channelporosity: {channelporosity}")

optpercolation: 1
channelconductivity: 6000 mm/hr = 0.0016666666666666668 m/s
channelporosity: 0.4


Lets specify some additonal parameter values for the tRIBS kinematic routing scheme

  - KINEMVELCOEF is a direct multiplier on hillslope velocity, doubling it halves hillslope travel time
  - FLOWEXP controls how much velocity scales with discharge, lower values mean more constant velocity regardless of storm intensity
  - CHANNELROUGHNESS is the Manning's n roughness coefficient for the channel
  - CHANNELWIDTHCOEFF controls the width of the channel network

We'll set KINEMVELCOEF and FLOWEXP to some standard starting values for tRIBS. Then set the roughness to value that is more aligned with what we saw in the field but it can still be adjusted.

In [307]:
# (TL) Routing settings for this calibration run.

model.kinemvelcoef['value'] = kinemvelcoef
model.flowexp['value'] = flowexp
model.channelroughness['value'] = channelroughness
model.channelwidthcoeff['value'] = channelwidthcoeff

print(f"kinemvelcoef: {kinemvelcoef}")
print(f"flowexp: {flowexp}")
print(f"channelroughness: {channelroughness}")
print(f"channelwidthcoeff: {channelwidthcoeff}")

kinemvelcoef: 2
flowexp: 0.3
channelroughness: 0.04
channelwidthcoeff: 2.33


In [308]:
# CHANGE THIS CELL TO CORRECT SNOW
# (TL) Sets main model-input keywords, including mesh source, 
# bedrock/land-use options, simulation start time and duration, 
# output naming, and rainfall time step. Notebook translating setup 
# choices into final tRIBS input-file values. 

# mesh
model.parallelmode['value'] = 0 # Running the model in serial mode, not parallel
model.optmeshinput['value'] = 1 # We are using mesh input option 1 for a pre-generated mesh
model.inputdatafile['value'] = "../smf_init_data/mesh/SMF_mesh" # filepath to our mesh files
model.inputtime['value'] = 0 # legacy option for tRIBS

# soil 
model.optbedrock['value'] = 1 # we are using our gridded bedrock depth map

# snow
model.optsnow['value'] = 0 # CORRECTION ADD THIS LINE TO TURN OFF SNOW MODULE

#land
model.optlanduse['value'] = 0 # Only paramters are from the land use table so option 0

# simulation variables
model.startdate['value'] = start_date
model.runtime['value'] = runtime_hours

# Send all run outputs to a run-specific folder/prefix
model.outfilename['value'] = output_prefix
model.outhydrofilename['value'] = output_prefix

# Forcings
model.rainintrvl['value'] = rain_interval_hours

print(f"Simulation start date: {start_date}")
print(f"Runtime hours: {runtime_hours}")
print(f"Rain interval hours: {rain_interval_hours}")
print(f"Output prefix: {output_prefix}")

Simulation start date: 08/01/2014/00/00
Runtime hours: 450
Rain interval hours: 0.25
Output prefix: ../calibration_work/02_results/40_multivariable/SMF_20140812_45_cc6000_cv2/SMF_20140812_45_cc6000_cv2


Below we need to specify which voronoi polygons we want to get additional detialed information on.

In [309]:
# (TL) Node inputs/outputs: writes node-list files for detailed polygon 
# and streamflow outputs, then links those files to the model.

# Create node list file for outputing Pixel file output (detailed timeseries for individual polygons)
node_ids = [1960, 1547, 3082] # these are example node IDs with no real meaning but can be changed to any ID value
model.write_node_file(node_ids,'data/model/pnodes.dat')
model.nodeoutputlist['value'] = 'data/model/pnodes.dat'

# Create node list file for outputing internal streamflow results (*.qout)
node_ids = [3202] # these are example node IDs with no real meaning but can be changed to any ID value
model.write_node_file(node_ids,'data/model/qnodes.dat')
model.outletnodelist['value'] = 'data/model/qnodes.dat'

In [310]:
# (TL) Validates file paths stored in model object before final input file 
# is written. Pre-run check.

model.check_paths()


The following tRIBS inputs do not have paths that exist: 


Checking if station descriptor paths exist.

All rain gauge paths exist.
All met station paths exist.

Checking if grid files exist.



In [311]:
# (TL) Define the run-specific tRIBS input file.

print(f"Input file for this run: {input_file}")

Input file for this run: ../calibration_work/01_run_inputs/40_multivariable/SMF_20140812_45_cc6000_cv2.in


In [312]:
# (TL) Writes the final tRIBS input file using all parameters, paths, 
# and options defined earlier in the notebook. 
# It is the main output of this notebook, and the file later used to run the model. 

model.write_input_file(input_file)